# MCMC exploration

Visual walk-through of `bayes_cep.mcmc`: builds the `LogPosterior` for the single retained patient (`example_data/raw/`, `example_data/preprocessing/`), then samples it with `MCMCBuilder` -- by default the classical pCN sampler, with MALA selectable via `MCMCSettings.algorithm`. Samples are written to a Zarr store.

The posterior is sampled relative to the prior as reference measure $\mu_0 = \mathcal N(\bar m, C)$, with the likelihood cost as potential $\Phi$ (see `bayes_cep.mcmc.measures`). The chain starts at the MAP estimate cached by `scripts/run_map.py` in `example_data/map/`.

This notebook is for interactive/visual inspection only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import zarr
from ls_bayesian.common.logging import BaseLogger, LoggerSettings
from ls_bayesian.mcmc.sampler import SamplerSettings

from bayes_cep.mcmc.builder import (
    MCMCAlgorithmName,
    MCMCBuilder,
    MCMCSettings,
    ZarrStorageSettings,
)
from bayes_cep.mesh.interpolation import NearestNeighborInterpolationStrategy
from bayes_cep.mesh.io import load_pyvista_mesh
from bayes_cep.posterior.builder import PosteriorBuilder, PosteriorSettings
from bayes_cep.posterior.eikonal_map import EikonalSolverSettings
from bayes_cep.posterior.likelihood import LikelihoodSettings
from bayes_cep.posterior.prior import PriorSettings
from bayes_cep.preprocessing.axial_statistics import compute_axial_data_diff

pv.set_jupyter_backend("trame")

DATA_DIR = Path.cwd().parent / "example_data"
RAW_DIR = DATA_DIR / "raw"
PREPROCESSING_DIR = DATA_DIR / "preprocessing"
MAP_DIR = DATA_DIR / "map"
MCMC_DIR = DATA_DIR / "mcmc"

## Load preprocessed data

In [ ]:
mesh = load_pyvista_mesh(RAW_DIR / "mesh.vtu")
basis_vectors = np.load(RAW_DIR / "basis_vecs.npy")
ground_truth_angle_field = np.load(PREPROCESSING_DIR / "ground_truth_angle_field.npy")
prior_mean_angle_field = np.load(PREPROCESSING_DIR / "prior_mean_angle_field.npy")
observed_vertex_indices = np.load(PREPROCESSING_DIR / "observed_vertex_indices.npy")
observed_activation_times = np.load(PREPROCESSING_DIR / "observed_activation_times.npy")
map_estimate = np.load(MAP_DIR / "map_estimate.npy")

interpolation_strategy = NearestNeighborInterpolationStrategy()
connectivity = mesh.cells.reshape(-1, 4)[:, 1:]
vertex_to_simplex_matrix = interpolation_strategy.assemble_matrix(mesh.points, connectivity)

print(f"{mesh.n_points} vertices, {mesh.n_cells} triangles")
print(f"{observed_vertex_indices.shape[0]} observed vertices")

## Build the posterior

Reference settings, matching `scripts/run_map.py`'s and `scripts/preprocess_data.py`'s defaults.

In [ ]:
posterior_settings = PosteriorSettings(
    mesh=mesh,
    basis_vectors=basis_vectors,
    prior_settings=PriorSettings(
        mean_vector=prior_mean_angle_field,
        kappa=0.05,
        tau=10.0,
        seed=0,
    ),
    eikonal_settings=EikonalSolverSettings(
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1.0,
    ),
    likelihood_settings=LikelihoodSettings(
        num_vertices=mesh.points.shape[0],
        observed_vertex_indices=observed_vertex_indices,
        data_vector=observed_activation_times,
        noise_variance=1e-3,
    ),
)
posterior_builder = PosteriorBuilder(
    posterior_settings, interpolation_strategy=interpolation_strategy
)
log_posterior = posterior_builder.build()

## Build the sampler

`MCMCSettings.algorithm` selects the sampler (`MCMCAlgorithmName.PCN` or `MCMCAlgorithmName.MALA`); both use `step_width`, with a default far smaller than usual since the likelihood is very sharp relative to the broad prior (see `bayes_cep.mcmc.builder`). `tracked_components` adds the value and running mean of single mesh vertices to the logged outputs, next to the running acceptance rate and the mean over all components, which are always logged. `MCMCBuilder` writes the samples to `MCMC_DIR / storage.store_name` and overwrites an existing store by default.

In [ ]:
mcmc_settings = MCMCSettings(
    algorithm=MCMCAlgorithmName.PCN,
    step_width=1e-4,
    tracked_components=(),
    storage=ZarrStorageSettings(chunk_size=100),
)
MCMC_DIR.mkdir(exist_ok=True)
mcmc_builder = MCMCBuilder(
    mcmc_settings, log_posterior, posterior_builder.prior, output_dir=MCMC_DIR
)

## Run the chain

> **Note:** every step needs a forward eikonal solve on the full mesh (MALA additionally an adjoint solve), so long chains take a long time. To only inspect the results of an earlier run, set `RUN_SAMPLER = False` in the next cell: it skips sampling and reads the existing Zarr store (the store is only created, and overwritten by default, when the sampler is built, i.e. when `RUN_SAMPLER` is `True`).

In [ ]:
RUN_SAMPLER = True
NUM_SAMPLES = 1000
SEED = 0

if RUN_SAMPLER:
    with BaseLogger(LoggerSettings(), prefix="mcmc") as logger:
        sampler_settings = SamplerSettings(
            num_samples=NUM_SAMPLES, log_interval=max(1, NUM_SAMPLES // 100)
        )
        sampler = mcmc_builder.build(logger=logger)
        sampler.run(
            initial_state=map_estimate,
            settings=sampler_settings,
            seed=SEED,
        )
    acceptance_rate_history = mcmc_builder.outputs[0].all_values
    mean_history = mcmc_builder.outputs[1].all_values
    print(f"final acceptance rate: {acceptance_rate_history[-1]:.3f}")

## Chain diagnostics

Running acceptance rate and the mean over all components of the state along the chain (only available if the chain was run in this session).

In [ ]:
if RUN_SAMPLER:
    fig, (acceptance_axis, mean_axis) = plt.subplots(1, 2, figsize=(10, 4))
    acceptance_axis.plot(acceptance_rate_history)
    acceptance_axis.set_xlabel("iteration")
    acceptance_axis.set_ylabel("running acceptance rate")
    mean_axis.plot(mean_history)
    mean_axis.set_xlabel("iteration")
    mean_axis.set_ylabel("mean over all components [rad]")
    fig.tight_layout()

## Posterior mean and variance

Computed block-wise from the Zarr store, so the full sample array never has to fit in memory. The first `BURN_IN` samples are discarded. The plain (not circular) mean and variance of the angle are used, which is adequate as long as the posterior stays concentrated away from the $\pi$-periodic wrap-around of the angle.

In [ ]:
BURN_IN = 100

samples = zarr.open_group(MCMC_DIR / mcmc_settings.storage.store_name, mode="r")["data"]
num_samples_total = samples.shape[0]
print(f"{num_samples_total} stored samples of dimension {samples.shape[1]}")

num_kept = num_samples_total - BURN_IN
sample_sum = np.zeros(samples.shape[1])
sample_square_sum = np.zeros(samples.shape[1])
for start in range(BURN_IN, num_samples_total, samples.chunks[0]):
    block = samples[start : min(start + samples.chunks[0], num_samples_total)]
    sample_sum += block.sum(axis=0)
    sample_square_sum += (block**2).sum(axis=0)
mcmc_mean = sample_sum / num_kept
mcmc_variance = sample_square_sum / num_kept - mcmc_mean**2
mcmc_std = np.sqrt(np.maximum(mcmc_variance, 0.0))

## Posterior mean vs. ground truth and MAP estimate

Compared on simplices (via `vertex_to_simplex_matrix`), as in `map.ipynb`: this is the representation the forward model actually sees.

In [ ]:
ground_truth_angle_field_simplex = vertex_to_simplex_matrix @ ground_truth_angle_field
map_estimate_simplex = vertex_to_simplex_matrix @ map_estimate
mcmc_mean_simplex = vertex_to_simplex_matrix @ mcmc_mean

for name, field_values in (
    ("ground truth", ground_truth_angle_field_simplex),
    ("MAP estimate", map_estimate_simplex),
    ("MCMC mean", mcmc_mean_simplex),
):
    plot_mesh = mesh.copy()
    plot_mesh.cell_data["angle"] = field_values
    plotter = pv.Plotter()
    plotter.add_mesh(plot_mesh, scalars="angle", cmap="hsv")
    plotter.add_text(name)
    plotter.show()

## Axial error vs. ground truth

As in `map.ipynb`, the signed axial difference (`compute_axial_data_diff`) is used, since the fiber angle is $\pi$-periodic.

In [ ]:
map_error = compute_axial_data_diff(map_estimate_simplex, ground_truth_angle_field_simplex)
mcmc_mean_error = compute_axial_data_diff(mcmc_mean_simplex, ground_truth_angle_field_simplex)
print(f"MAP estimate mean absolute error: {np.mean(np.abs(map_error)):.4f} rad")
print(f"MCMC mean mean absolute error:    {np.mean(np.abs(mcmc_mean_error)):.4f} rad")

plot_mesh = mesh.copy()
plot_mesh.cell_data["error"] = mcmc_mean_error
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="error", cmap="coolwarm", clim=(-np.pi / 2, np.pi / 2))
plotter.add_text("MCMC mean error")
plotter.show()

## Posterior standard deviation

In [ ]:
plot_mesh = mesh.copy()
plot_mesh.point_data["std"] = mcmc_std
plotter = pv.Plotter()
plotter.add_mesh(plot_mesh, scalars="std", cmap="viridis")
plotter.add_text("MCMC pointwise standard deviation [rad]")
plotter.show()